# 20 — Direction 1: replica caches, the baseline's null group, and V2's generated scenarios

Per scenario, on a CPU:

1. **cache** (`hvac_controller/d1/cache.py`): spec, ROM, exogenous series, policy metadata and action spec, pickled
   once so that training and evaluation sessions never rebuild the ROM (≈ 1–4 min each). Also logs the Jacobi bound q
   and the float32 sweep count (must stay < 0.5 for the fast air solve).
2. **original-simulator runs** (`hvac_controller/d1/evaluate.py`): reference policies for 30 days in the organisers'
   simulator.

Modes:
- **`MODE = "pilot"`**: atlanta_2 only, K10 with seeds 0–3 in the original and the replica, ≈ 15–20 min.
- **`MODE = "all"`**: the 30 shipped scenarios, same runs, `N_SHARDS = 3` sessions with `RUN_ID = 0 … 2`.
- **`MODE = "gen"` (V2)**: the generated scenarios of `hvac_controller/d1/augment.py`.
  - 194 scenarios: 60 generated buildings × 3 TMYx strips (training), 6 held-out buildings × 2 strips of held-out
    stations, and building_train_2 × minneapolis_1/_5.
  - `N_SHARDS = 4` sessions in parallel, `RUN_ID = 0 … 3`, each ≈ 1.5–2.5 h.
  - Each session builds the same manifest and buildings, then its share of the weather presets, scenarios and
    caches.
  - It then runs K10 and V1 (the submitted package, run with its own code) in the original simulator, seed 0, on its
    share of the 14 held-out scenarios.
- **`MODE = "fill"`**: the generated caches the V2 data dataset still lacks (`gen` built 127 of 194 on 2026-10-21:
  3 builds at once ran out of RAM on the large buildings).
  - Reads `manifest_v2.json`, the buildings' YAMLs and the caches from that dataset; builds the missing caches one at a
    time (`N_JOBS = 1`, all 4 CPUs); no simulator runs.
  - `N_SHARDS = 4` sessions in parallel, `RUN_ID = 0 … 3`, each ≈ 2–3 h.

**Setup:** CPU; Internet on (clones the repo); inputs: the competition data, and for `gen` also the TMYx weather
dataset (the packed `data.zip`) and the V1 submission dataset (`d1_policy.py`, `d1_policy.pt`); for `fill` the TMYx
weather dataset and the V2 cache dataset (`data/d1_cache_v2.zip`, with `manifest_v2.json`).

**Output:**
- `pilot` / `all`: `/kaggle/working/d1_cache_<mode>_<run>.zip`, with logs and results.
- `gen`: `/kaggle/working/d1_cache_gen_<run>.zip`, **flat**: the caches (`*.pkl` + `*.json`), the generated buildings'
  YAMLs, the held-out scenarios' weather presets (`x*.yaml` + `x*.csv`), the manifest, the original-simulator
  results and the logs. Download the 4 zips; they merge into the dataset notebook 21 attaches.
- `fill`: `/kaggle/working/d1_cache_fill_<run>.zip`, flat: the new caches (`*.pkl` + `*.json`), their logs and the summary.
  Download the 4 zips; they merge into that dataset's next version.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
MODE = "fill"             # "pilot", "all", "gen" or "fill"
RUN_ID, N_SHARDS = 0, 4   # all: 3 shards; gen, fill: 4 shards
PILOT = ["building_train_1/atlanta_2"]
SEEDS = 4                 # pilot / all: seed 0 plain + 3 noisy runs per scenario (gen: seed 0 only)
N_PROCS = 4               # simulator runs in parallel (4 CPUs)
N_JOBS = 1 if MODE == "fill" else 3  # cache builds in parallel (fill: the large buildings, 3 at once ran out of RAM)

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO = Path("/tmp/hvac-controller")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, str(REPO))

SHIPPED = next(p for p in sorted(Path("/kaggle/input").rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
OUT = Path("/kaggle/working/d1_cache")
CACHES, EVAL, LOGS = OUT / "caches", OUT / "eval", OUT / "logs"
for d in (CACHES, EVAL, LOGS):
    d.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(REPO))
VAL_TAGS = ("gen_val", "shipped_building_new_station")

if MODE in ("gen", "fill"):
    from hvac_controller import weather

    PACKED = weather.find_packed(Path("/kaggle/input"))
    SHARD = OUT / f"shard_{RUN_ID}.json"
    if MODE == "gen":
        V1 = Path("/kaggle/working/v1")  # the V1 package, run with its own code; "v1" names its results
        src = next(p.parent for p in sorted(Path("/kaggle/input").rglob("d1_policy.py")) if (p.parent / "d1_policy.pt").exists())
        shutil.copytree(src, V1, dirs_exist_ok=True)
        print("weather:", PACKED, "| V1:", src, (src / "README.txt").read_text() if (src / "README.txt").exists() else "")
        MANIFEST = OUT / "manifest_v2.json"
        steps = [["manifest", str(MANIFEST), "--packed", str(PACKED)], ["buildings", str(MANIFEST)]]
    else:
        DATA = next(p.parent for p in sorted(Path("/kaggle/input").rglob("manifest_v2.json")))
        for p in DATA.glob("building_0*.yaml"):
            shutil.copy(p, REPO / "smart_buildings" / "buildings" / p.name)
        done = {p.stem for p in DATA.glob("*.pkl")}
        todo = [e for e in json.loads((DATA / "manifest_v2.json").read_text()) if e["label"] not in done]
        MANIFEST = OUT / "manifest_fill.json"
        MANIFEST.write_text(json.dumps(todo, indent=1))
        print("weather:", PACKED, "| data:", DATA, f"| {len(done)} caches there, {len(todo)} generated ones missing")
        steps = []
    t0 = time.time()
    for cmd in steps + [["scenarios", str(MANIFEST), "--packed", str(PACKED), "--shard", str(RUN_ID),
                         "--shards", str(N_SHARDS), "--out", str(SHARD)]]:
        r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.augment", *cmd], cwd=REPO, env=ENV,
                           capture_output=True, text=True)
        print(f"[{time.time() - t0:5.0f}s] augment {cmd[0]}: exit {r.returncode}\n" + (r.stdout + r.stderr)[-3000:])
        assert r.returncode == 0, cmd
    ENTRIES = json.loads(SHARD.read_text())
    SCENARIOS = [e["label"].replace("__", "/", 1) for e in ENTRIES]
    VAL_HERE = [e["label"].replace("__", "/", 1) for e in ENTRIES if e["tag"] in VAL_TAGS]
else:
    ALL = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
    SCENARIOS = PILOT if MODE == "pilot" else ALL[RUN_ID::N_SHARDS]
    assert set(SCENARIOS) <= set(ALL), sorted(set(SCENARIOS) - set(ALL))
missing = [s for s in SCENARIOS if not (ROOT / s / "config.gin").exists()]
assert not missing, missing
print(len(SCENARIOS), "scenarios here:", SCENARIOS[:6], "...")

In [ ]:
# 1. Caches, N_JOBS at a time.
threads = str(max(1, (os.cpu_count() or 4) // N_JOBS))
env = dict(ENV, OMP_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads, MKL_NUM_THREADS=threads)
t0 = time.time()
pending, running = list(SCENARIOS), []
while pending or running:
    while pending and len(running) < N_JOBS:
        scenario = pending.pop(0)
        tag = scenario.replace("/", "__")
        log = open(LOGS / f"cache__{tag}.txt", "w")
        cmd = [sys.executable, "-m", "hvac_controller.d1.cache", str(CACHES / f"{tag}.pkl"),
               "--config", str(ROOT / scenario / "config.gin")]
        running.append((tag, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    for item in list(running):
        tag, proc, log = item
        if proc.poll() is not None:
            log.close()
            running.remove(item)
            text = (LOGS / f"cache__{tag}.txt").read_text()
            print(f"[{time.time() - t0:5.0f}s] {tag} exit {proc.returncode}: {text.strip()[-1500:]}", flush=True)
    time.sleep(5)

In [ ]:
# 2. Reference policies in the original simulator (pilot / all: K10, also in the replica, SEEDS runs each;
#    gen: K10 and V1, seed 0, on this shard's held-out scenarios; fill: none, the dataset has them).
if MODE == "gen":
    RUN_SCENARIOS, POLICIES, N_SEEDS, WHERE = VAL_HERE, f"k10,pkg:{V1}", 1, "original"
elif MODE == "fill":
    RUN_SCENARIOS, POLICIES, N_SEEDS, WHERE = [], "k10", 1, "original"
else:
    RUN_SCENARIOS, POLICIES, N_SEEDS, WHERE = SCENARIOS, "k10", SEEDS, "original,replica"
if RUN_SCENARIOS:
    cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(RUN_SCENARIOS),
           "--scenario-root", str(ROOT), "--cache-dir", str(CACHES), "--policies", POLICIES, "--seeds", str(N_SEEDS),
           "--where", WHERE, "--n-procs", str(N_PROCS)]
    with open(LOGS / f"eval_{MODE}_{RUN_ID}.txt", "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    print("exit", proc.wait())

In [ ]:
# 3. Summary, gate and the zip.
infos = {p.stem: json.loads(p.read_text()) for p in sorted(CACHES.glob("*.json"))}
results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
(OUT / f"summary_{MODE}_{RUN_ID}.json").write_text(json.dumps({"commit": COMMIT, "mode": MODE, "run_id": RUN_ID,
                                                               "caches": infos}, indent=1, default=str))
for tag, i in infos.items():
    print(f"{tag:36s} Z={i['n_zones']:3d} V={i['n_vavs']:3d} A={i['n_ahus']:2d} obs={i['n_obs']} r={i['order']} "
          f"h={i['h']:.2f} q={i['jacobi_q']:.3f} sweeps={i['sweeps_float32']} | ROM {i['rom_s']:.0f}s exo {i['exo_s']:.0f}s")
errors = [r for r in results if "error" in r]
for r in errors:
    print("ERROR", r["where"], r["scenario"], r["policy"], r["seed"], r["error"])
by = {}
for r in results:
    if "error" not in r:
        by.setdefault((r["scenario"], r["where"], r["policy"].split("/")[-1]), []).append(r["score"])
for key, v in sorted(by.items()):
    print(f"{key[0]:36s} {key[1]:8s} {key[2]:10s} {sum(v) / len(v):.5f} (n={len(v)})")
n_pol = len(POLICIES.split(",")) * len(WHERE.split(","))
ok = (all(i["jacobi_q"] < 0.5 for i in infos.values()) and len(infos) == len(SCENARIOS) and not errors
      and len([r for r in results if "error" not in r]) == len(RUN_SCENARIOS) * n_pol * N_SEEDS)
print(f"\nGATE: {len(infos)}/{len(SCENARIOS)} caches, all q < 0.5; {len(results)} runs "
      f"(want {len(RUN_SCENARIOS) * n_pol * N_SEEDS}), no errors:", ok)

name = f"/kaggle/working/d1_cache_{MODE}_{RUN_ID}.zip"
if MODE == "gen":
    files = (list(CACHES.glob("*.pkl")) + list(CACHES.glob("*.json")) + list(LOGS.glob("*")) + list(EVAL.glob("*.json"))
             + [MANIFEST, SHARD, OUT / f"summary_{MODE}_{RUN_ID}.json"]
             + sorted((REPO / "smart_buildings" / "buildings").glob("building_0*.yaml")))
    for e in ENTRIES:  # the held-out scenarios' presets: notebook 21 rebuilds those scenarios for the original simulator
        if e["tag"] in VAL_TAGS and "stem" in e:
            files += [REPO / "smart_buildings" / "weather" / f"{e['weather']}.yaml",
                      REPO / "smart_buildings" / "weather" / "csv" / f"{e['weather']}.csv"]
    names = [p.name for p in files]
    assert len(names) == len(set(names)), "flat zip: duplicate file names"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.name)
elif MODE == "fill":  # names as in the dataset: logs prefixed log_fill<run>_
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in list(CACHES.glob("*.pkl")) + list(CACHES.glob("*.json")) + [OUT / f"summary_{MODE}_{RUN_ID}.json"]:
            z.write(p, p.name)
        for p in LOGS.glob("*"):
            z.write(p, f"log_fill{RUN_ID}_{p.name}")
else:
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in list(LOGS.glob("*")) + list(EVAL.glob("*.json")) + list(CACHES.glob("*.json")) + [OUT / f"summary_{MODE}_{RUN_ID}.json"]:
            z.write(p, p.relative_to(OUT.parent))
print("wrote", name, f"{Path(name).stat().st_size / 2 ** 20:.0f} MB")